# Week 3 — Optimisation, Likelihood and Maximum-Likelihood Estimation

**Contact:** 3 hours lecture  
**Course:** MSc Astrostatistics for the Survey Era

## Learning notebooks from the Crete school
- `../../03_Optimization/Optimization.ipynb`
- `../../04_MLE/MLE.ipynb`

These Crete notebooks are **learning material**. This lecture notebook supplies the semester narrative, deeper astronomical interpretation, and the binary-SMBH case study.

## Three-hour pacing
- **0:00–0:20** — recap, motivating scientific question, diagnostic poll
- **0:20–1:05** — theory block I + worked derivation
- **1:05–1:15** — short break / discussion
- **1:15–2:00** — theory block II + computational demonstration
- **2:00–2:10** — short break
- **2:10–2:45** — binary-SMBH case study / failure mode
- **2:45–3:00** — synthesis, exit questions, bridge to the 4-hour practical

## Learning objectives

- Construct a likelihood from a generative measurement model.
- Use log-likelihoods and numerical optimisation safely.
- Understand MLE invariance, curvature, profile likelihoods and bootstrap uncertainty.
- Recognise boundaries, degeneracies and model misspecification.
- Fit simple SMBH population models while keeping simulation priors distinct from cosmic population claims.

## 1. From a forward model to a likelihood


A likelihood is not simply a convenient loss function. It is the probability model for the observed data viewed as a function of unknown parameters:

\[
\mathcal L(\theta;D)=p(D\mid\theta).
\]

For independent Gaussian measurements \(y_i\) with known \(\sigma_i\),

\[
\log\mathcal L(\theta)=
-\frac12\sum_i\left[\frac{(y_i-\mu_i(\theta))^2}{\sigma_i^2}
+\log(2\pi\sigma_i^2)\right].
\]

Dropping terms that do not depend on \(\theta\) may be safe for point optimisation but not always for comparing models whose normalisations differ.

Astronomical likelihoods may be Poisson for counts, Gaussian for high-S/N approximately symmetric measurements, binomial for detections, or much more structured for time series. The likelihood must describe what is **observed**, not what would be convenient to optimise.

> **Discussion checkpoint.** Identify the estimand, observed data, and strongest modelling assumption in the example just discussed.

### Worked computational demonstration — Gaussian log-likelihood for log mass

Run the cell, then change one assumption and explain what changes statistically.

In [ ]:
import numpy as np, sys
from scipy.optimize import minimize
sys.path.append('../src')
from course_utils import load_scalar_sample
df=load_scalar_sample()
y=np.log10(df['inputs__total_mass'].to_numpy())

def nll(par):
    mu, log_sigma=par
    sigma=np.exp(log_sigma)
    return 0.5*np.sum(((y-mu)/sigma)**2 + 2*np.log(sigma) + np.log(2*np.pi))

res=minimize(nll,[np.mean(y),np.log(np.std(y))])
mu_hat, sigma_hat=res.x[0],np.exp(res.x[1])
print(res.success, mu_hat, sigma_hat)

## 2. Optimisation and parameterisation


The MLE is

\[
\hat\theta_{\rm ML}=\arg\max_\theta \log\mathcal L(\theta).
\]

Optimisers see geometry, not physics. Poor parameterisation can create steep boundaries, highly elongated valleys or invalid trial values. Positive parameters are often more stable in log space. Fractions can be represented by logits; ordered parameters may need constrained transformations.

For SMBH total mass and period, log transforms are natural because the ranges span orders of magnitude. When reporting results, however, remember that a symmetric interval in log space becomes asymmetric on the physical scale.

### Worked computational demonstration — Likelihood surface

Run the cell, then change one assumption and explain what changes statistically.

In [ ]:
import matplotlib.pyplot as plt
mus=np.linspace(mu_hat-.25,mu_hat+.25,120)
sigs=np.linspace(.5*sigma_hat,1.5*sigma_hat,120)
Z=np.empty((len(sigs),len(mus)))
for i,s in enumerate(sigs):
    for j,m in enumerate(mus):
        Z[i,j]=nll([m,np.log(s)])
fig,ax=plt.subplots()
cs=ax.contour(mus,sigs,Z-Z.min(),levels=[1.15,3.0,6.0,10.0])
ax.set(xlabel='mu(logM)',ylabel='sigma(logM)')
ax.clabel(cs)
plt.show()

## 3. Curvature, profile likelihood and uncertainty


Near a well-behaved maximum, the log-likelihood can be approximated by a quadratic. In one dimension,

\[
-\frac{\partial^2\log\mathcal L}{\partial\theta^2}
\]

provides local information about uncertainty. In multiple dimensions the Hessian reveals covariance and degeneracy.

This approximation can fail near boundaries, with small samples, with heavy tails or multimodal likelihoods. Profile likelihoods are often more informative: for a parameter of interest \(\psi\), maximise over nuisance parameters \(\lambda\),

\[
\mathcal L_p(\psi)=\max_\lambda \mathcal L(\psi,\lambda).
\]

Bootstrap distributions provide another route to finite-sample uncertainty and are an excellent diagnostic when asymptotic approximations are doubtful.

### Worked computational demonstration — Parametric bootstrap

Run the cell, then change one assumption and explain what changes statistically.

In [ ]:
rng=np.random.default_rng(2026)
mu_boot=[]
for _ in range(1000):
    ys=rng.normal(mu_hat,sigma_hat,len(y))
    mu_boot.append(np.mean(ys))
print('95% parametric bootstrap CI:',np.quantile(mu_boot,[.025,.975]))

## 4. Misspecification: the best wrong model


A maximum-likelihood estimate is optimal only relative to the assumed model. If the true distribution is not in the model family, the optimiser still returns the parameter value that best compromises under that family.

This is particularly important for simulated population distributions. A smooth log-normal may approximate the simulated total-mass distribution, yet the simulator could have been generated by a different prior or a mixture. Students should inspect residuals, empirical CDFs and predictive simulations instead of stopping at successful convergence.

The relevant question is not "did the optimiser return success=True?" but "does the fitted model reproduce the aspects of the data that matter for the scientific claim?"

## 5. Case study: fitting a transformed SMBH population


A useful controlled exercise is to fit a Gaussian model to \(y=\log_{10}M_{\rm tot}\). The likelihood is straightforward, the MLE has an analytic reference solution, and the same problem can be revisited in Week 4 with priors and posterior prediction.

This continuity is deliberate. Students will see that MLE, Bayesian inference and MCMC are not unrelated algorithms; they are different computational and inferential treatments of the same generative model.

## Deep dive — identifiability, nuisance parameters and heteroscedastic likelihoods

A parameter is identifiable if distinct parameter values imply distinguishable data distributions. Optimisation cannot rescue a non-identifiable model. A flat or elongated likelihood valley may therefore be a scientific feature rather than a numerical failure.

Consider heteroscedastic measurements

\[
y_i\sim N(\mu(x_i;\theta),\sigma_i^2+\sigma_{\rm int}^2),
\]

where \(\sigma_i\) is known measurement uncertainty and \(\sigma_{\rm int}\) is unknown intrinsic scatter. The log-likelihood includes both residual and normalisation terms:

\[
\log\mathcal L=
-\frac12\sum_i
\left[
\frac{(y_i-\mu_i)^2}{\sigma_i^2+\sigma_{\rm int}^2}
+\log(\sigma_i^2+\sigma_{\rm int}^2)
+\log(2\pi)
\right].
\]

If the logarithmic variance term is dropped, the fit can artificially inflate \(\sigma_{\rm int}\) to reduce the residual penalty.

### Nuisance parameters

A nuisance parameter is not unimportant; it is simply not the main scientific target. Calibration offsets, noise amplitudes or population scatter may be nuisance quantities that must still be modelled to obtain a valid target estimate.

MLE can handle nuisance parameters by joint optimisation or profile likelihood. Bayesian analysis will instead integrate/marginalise them. This difference becomes important in the next two weeks.

### Boundary problems

Mass ratio is bounded. Eccentricity is bounded. Variance parameters are positive. Near boundaries, Gaussian curvature approximations for uncertainty can be poor. Reparameterisation—e.g. log variance, logit fractions—often helps numerically but does not remove physical boundary effects.

A high-quality fit therefore reports the optimisation method, parameterisation, initialisation checks, and at least one diagnostic of the likelihood shape.

## Binary-SMBH synthesis questions

Answer verbally before leaving the lecture.

1. Which quantity in today's topic is a **property of the simulator**, and which could be an **observable property of a real survey**?
2. What uncertainty or selection effect would most strongly distort the result?
3. What diagnostic would you require before trusting the inference?
4. What information would be unavailable if these were real LSST sources rather than simulations?

## Key equations / ideas to retain

Create your own 6–10 line summary here during class.

## Further reading

- Crete `03_Optimization` and `04_MLE` notebooks.
- Ivezić et al., likelihood and optimisation chapters.
- Feigelson & Babu, estimation chapters.

# Part II — Extended lecture development

## 13. Ordinary least squares derived geometrically

For \(y_i=a+bx_i+\epsilon_i\), minimise

\[
S(a,b)=\sum_i(y_i-a-bx_i)^2.
\]

The solution is

\[
\hat b=
\frac{\sum_i(x_i-\bar x)(y_i-\bar y)}
{\sum_i(x_i-\bar x)^2},
\qquad
\hat a=\bar y-\hat b\bar x.
\]

Thus OLS slope is covariance divided by predictor variance. It also reveals a
key assumption: \(x\) is treated as known. Measurement error in \(x\) requires
an errors-in-variables model.

## 14. Weighted least squares

For known unequal uncertainties,

\[
y_i\sim N(a+bx_i,\sigma_i^2),
\]

MLE minimises

\[
\chi^2=\sum_i\frac{[y_i-(a+bx_i)]^2}{\sigma_i^2}.
\]

Small-error points carry more information. But weights must represent the
variance model, not subjective importance. Incorrect \(\sigma_i\) values imply
incorrect formal parameter uncertainty.

## 15. Likelihood-ratio geometry

Contours of

\[
-2[\ell(\theta)-\ell(\hat\theta)]
\]

visualise uncertainty and degeneracy. Elongated contours indicate covariance;
curved ridges reveal nonlinear degeneracy that a Hessian can only approximate
locally.

Low-dimensional likelihood plots are therefore diagnostic, not decorative.

In [ ]:
y=np.log10(df["inputs__total_mass"].astype(float).to_numpy())
mu0=y.mean(); s0=y.std(ddof=0)
mus=np.linspace(mu0-.2,mu0+.2,80)
sigs=np.linspace(.75*s0,1.25*s0,80)
LL=np.empty((len(sigs),len(mus)))
for i,s in enumerate(sigs):
    for j,m in enumerate(mus):
        LL[i,j]=np.sum(stats.norm.logpdf(y,loc=m,scale=s))
d=-2*(LL-LL.max())
plt.contour(mus,sigs,d,levels=[2.30,6.17,11.8])
plt.xlabel("mu(logM)"); plt.ylabel("sigma(logM)")
plt.show()

## 16. Optimisation diagnostics

Before trusting an optimum, check convergence status, gradient where available,
multiple starting values, boundaries, physical validity and alternative
solutions. Multiple random starts are a cheap way to expose local minima.

"Optimisation succeeded" is a computational statement, not evidence that the
model is scientifically adequate.

## 17. Robust likelihoods

Gaussian residual likelihoods impose a quadratic penalty. Student-\(t\)
likelihoods have heavier tails:

\[
p(r)\propto
\left(1+\frac{r^2}{\nu s^2}\right)^{-(\nu+1)/2}.
\]

They can reduce sensitivity to outliers without deleting points. But if extreme
objects represent a separate population, a mixture model may be more
scientifically meaningful than simply downweighting them.

In [ ]:
r=np.linspace(-6,6,500)
g=-stats.norm.logpdf(r)
t=-stats.t.logpdf(r,df=3)
plt.plot(r,g-g.min(),label="Gaussian")
plt.plot(r,t-t.min(),label="Student-t")
plt.xlabel("standardized residual"); plt.ylabel("relative negative log density")
plt.legend(); plt.show()

## 18. Information criteria

Common maximum-likelihood criteria include

\[
{\rm AIC}=2k-2\ell_{\max},
\qquad
{\rm BIC}=k\log n-2\ell_{\max}.
\]

They penalise complexity differently and rely on approximations. They should not
replace predictive checks or scientific reasoning. A model can win AIC while
failing an astrophysically important tail.

## 19. A correct optimiser on the wrong likelihood is still wrong

Selection, correlated noise, wrong error bars or an inappropriate population
family can all invalidate the likelihood. MLE does not automatically remove
bias from these modelling mistakes.

This point motivates Week 6: selection must appear explicitly in the population
likelihood.

## 20. Three levels of a fitting claim

Keep separate:

1. **Computational:** the optimiser found a numerical maximum.
2. **Statistical:** under the assumed likelihood this is the MLE with stated
   uncertainty.
3. **Scientific:** the model is an adequate representation for the intended
   claim.

Only diagnostics and validation can support the third statement.

# Part III — Worked board problems and derivations

## Problem A — derive Gaussian MLE for the mean

For independent

\[
y_i\sim N(\mu,\sigma^2)
\]

with known \(\sigma\),

\[
\ell(\mu)=
-\frac{1}{2\sigma^2}\sum_i(y_i-\mu)^2+\mathrm{const}.
\]

Differentiate:

\[
\frac{d\ell}{d\mu}
=
\frac1{\sigma^2}\sum_i(y_i-\mu).
\]

Set to zero:

\[
\hat\mu=\bar y.
\]

The second derivative is \(-n/\sigma^2<0\), confirming a maximum.

This simple derivation is useful because it connects a familiar estimator to a
generative probability model. The sample mean is not universally "the best
estimate"; it is the MLE under a specific likelihood.

## Problem B — unknown intrinsic scatter

Suppose

\[
y_i\sim N(\mu,\sigma_i^2+s^2),
\]

with known measurement errors \(\sigma_i\) and unknown intrinsic scatter \(s\).

Ask students to write the full negative log likelihood and identify why both
terms

\[
\frac{(y_i-\mu)^2}{\sigma_i^2+s^2}
\quad\text{and}\quad
\log(\sigma_i^2+s^2)
\]

must appear.

Then ask: what happens if all \(\sigma_i\) are multiplied by two? Which part of
the inference changes? Could \(s\) compensate? This opens discussion of
identifiability between measurement uncertainty and intrinsic scatter.

## Problem C — likelihood ratio near a boundary

Suppose the parameter is an amplitude constrained to \(A\ge0\), and the MLE is
at \(A=0\).

The usual symmetric Gaussian/Hessian uncertainty is immediately suspicious
because half of that approximation lies in the forbidden region. Regular
chi-square likelihood-ratio asymptotics can also change at boundaries.

The correct response is not to force a symmetric error bar. Use a method
appropriate to the boundary: profile likelihood with correct calibration,
simulation, or Bayesian posterior with an explicit non-negative prior.

In [ ]:
# Visual demonstration of a boundary-constrained likelihood
A=np.linspace(0,3,500)
# pretend unconstrained estimate would have been -0.3 +/- 0.6
logL=-.5*((A+0.3)/.6)**2
logL-=logL.max()
plt.plot(A,np.exp(logL))
plt.xlabel("A >= 0"); plt.ylabel("relative likelihood")
plt.title("Likelihood maximum sits at physical boundary")
plt.show()

# Part IV — Additional in-class derivation

## The score and Fisher-information idea

For log-likelihood \(\ell(\theta)\), the **score** is

\[
U(\theta)=\frac{\partial\ell}{\partial\theta}.
\]

At an interior MLE the score is zero. The curvature describes how quickly the
likelihood falls away from the optimum.

The Fisher information is

\[
I(\theta)
=
-E\left[
\frac{\partial^2\ell}{\partial\theta^2}
\right].
\]

Under regular large-sample conditions,

\[
\hat\theta_{\rm ML}
\approx
N\left(\theta,\ I(\theta)^{-1}\right).
\]

This connects likelihood curvature, information and asymptotic uncertainty.
More observations usually increase information, but correlated observations do
not add information as quickly as independent ones.

### Discussion

If every point in a light curve were duplicated ten times, the row count would
increase by a factor ten. Would the physical information increase tenfold?
No: exact duplicates add no independent information. The likelihood/covariance
model must represent dependence correctly.

This motivates two habits: count independent information, not rows, and inspect
likelihood geometry rather than quoting optimiser errors uncritically.